# Extract Model IDs where C10 = 'MRM Attachments'
Model ID is in cell **B4**, condition is checked in cell **C10**.

In [ ]:
import openpyxl
import glob
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
FOLDER_PATH = r'C:\Your\Folder\Path'          # <-- change to your folder
SHEET_NAME  = 'General Model & Review Summary' # <-- change if sheet name differs
MODEL_ID_CELL = 'B4'
MDD_CELL      = 'C10'
MRM_VALUE     = 'MRM Attachments'
# ─────────────────────────────────────────────────────────────────────────────

excel_files = glob.glob(os.path.join(FOLDER_PATH, '*.xlsx')) + \
              glob.glob(os.path.join(FOLDER_PATH, '*.xlsm'))

print(f'Found {len(excel_files)} Excel file(s)\n')

matched = []
errors  = []

for filepath in excel_files:
    filename = os.path.basename(filepath)
    try:
        wb = openpyxl.load_workbook(filepath, data_only=True)
        
        if SHEET_NAME not in wb.sheetnames:
            errors.append((filename, f'Sheet "{SHEET_NAME}" not found. Available: {wb.sheetnames}'))
            continue
        
        ws = wb[SHEET_NAME]
        
        c10_value = ws[MDD_CELL].value
        b4_value  = ws[MODEL_ID_CELL].value
        
        # Normalize and compare
        if c10_value and str(c10_value).strip().lower() == MRM_VALUE.lower():
            matched.append({
                'Model ID'   : str(b4_value).strip() if b4_value else 'BLANK',
                'C10 Value'  : str(c10_value).strip(),
                'Source File': filename
            })
    
    except Exception as e:
        errors.append((filename, str(e)))

# ── Print Results ─────────────────────────────────────────────────────────────
print(f'=== {len(matched)} file(s) matched (C10 = "{MRM_VALUE}") ===')
for row in matched:
    print(f"  Model ID: {row['Model ID']}  |  File: {row['Source File']}")

if errors:
    print(f'\n=== {len(errors)} file(s) with errors ===')
    for fname, err in errors:
        print(f'  {fname}: {err}')

In [ ]:
# ── Optional: Export to Excel ─────────────────────────────────────────────────
import pandas as pd

if matched:
    results_df  = pd.DataFrame(matched)
    output_path = os.path.join(FOLDER_PATH, 'MRM_Attachment_ModelIDs.xlsx')
    results_df.to_excel(output_path, index=False)
    print(f'Saved {len(matched)} record(s) to: {output_path}')
    display(results_df)
else:
    print('No matches to export.')

# Plain list of Model IDs
model_id_list = [r['Model ID'] for r in matched]
print('\nModel ID list:', model_id_list)